# Top 25 Widest Text Similarity Gaps: Deepgram vs Genesys

Extracts the 25 utterance pairs with the lowest text similarity
between Deepgram Nova-3 and Genesys r2d2 transcriptions of the same audio.
Outputs a markdown document to `analysis_results/`.

In [1]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
EB_CORRELATION_CSV = REPO_ROOT / "analysis_results" / "cross_system_eb" / "eb_correlation.csv"
OUTPUT_DIR = REPO_ROOT / "analysis_results"
OUTPUT_FILE = OUTPUT_DIR / "text_similarity_gap_examples.md"

MAX_EXAMPLES = 25

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source: {EB_CORRELATION_CSV}")
print(f"Output: {OUTPUT_FILE}")

Source: /Users/xnxn040/PycharmProjects/notifications-spike/analysis_results/cross_system_eb/eb_correlation.csv
Output: /Users/xnxn040/PycharmProjects/notifications-spike/analysis_results/text_similarity_gap_examples.md


In [2]:
df = pd.read_csv(EB_CORRELATION_CSV)

# Filter out negative-latency false matches
df = df[df["true_latency_ms"] >= 0].copy()

df["similarity_gap"] = 1.0 - df["similarity"]

print(f"Total matched utterance pairs: {len(df)}")
print(f"Similarity range: {df['similarity'].min():.1%} – {df['similarity'].max():.1%}")
print(f"Mean similarity: {df['similarity'].mean():.1%}")

Total matched utterance pairs: 62
Similarity range: 70.6% – 100.0%
Mean similarity: 87.2%


In [3]:
top_gaps = df.sort_values("similarity", ascending=True).head(MAX_EXAMPLES).copy()

print(f"Examples to include: {len(top_gaps)}")
print(f"Similarity range in selection: {top_gaps['similarity'].min():.1%} – {top_gaps['similarity'].max():.1%}")
print(f"Mean similarity gap in selection: {top_gaps['similarity_gap'].mean():.1%}")

Examples to include: 25
Similarity range in selection: 70.6% – 84.6%
Mean similarity gap in selection: 23.0%


In [4]:
top_gaps[
    ["deepgram_transcript", "genesys_transcript", "similarity", "similarity_gap",
     "deepgram_confidence", "genesys_confidence"]
].reset_index(drop=True)

,deepgram_transcript,genesys_transcript,similarity,similarity_gap,deepgram_confidence,genesys_confidence
0,Truth is,true dish,0.706,0.294,0.993,0.671
1,I don't get this yet.,i don't get a,0.710,0.290,0.969,0.712
2,Who has had the unmitigated terminology,who has had the unheated ten years,0.712,0.288,0.959,0.707
3,"Don't do likewise, gents.",don't do like chan,0.718,0.282,0.926,0.699
4,All Negro men are not to be trusted around our...,oh nibra bend that to be trusted around our wi...,0.725,0.275,0.986,0.699
5,front of you. I figure I'll stick to your card...,trying to do it figure out to your card this time,0.731,0.269,0.977,0.762
6,"I I I'm just not the the hero type, clearly.",fantastic i'm just not the hero type clearly w...,0.745,0.255,0.998,0.859
7,"This fellow will grow tired. Oh, he blows his ...",limpstein of yours my left footage fellow will...,0.746,0.254,0.988,0.866
8,It is one thing when you question the official...,it is one thing question the official story an...,0.748,0.252,0.993,0.858
9,How do you drink with such a nose? You must ha...,practical how do you drink with such a nose yo...,0.749,0.251,0.993,0.815


In [5]:
lines = [
    "# Top 25 Widest Text Similarity Gaps: Deepgram vs Genesys\n",
    "",
    "**Metric:** Text similarity (SequenceMatcher ratio) between Deepgram Nova-3 and Genesys r2d2 transcriptions of the same audio segment.\n",
    f"**Source:** `cross_system_eb/eb_correlation.csv` ({len(df)} matched utterance pairs)\n",
    f"**Shown:** {len(top_gaps)} utterance pairs with the lowest text similarity\n",
    f"**Mean similarity (all pairs):** {df['similarity'].mean():.1%}\n",
    f"**Mean similarity (these {len(top_gaps)}):** {top_gaps['similarity'].mean():.1%}\n",
    "",
    "---\n",
    "",
]

for i, (_, row) in enumerate(top_gaps.iterrows(), start=1):
    lines.append(f"## Example {i}\n")
    lines.append("")
    lines.append("| Metric | Value |")
    lines.append("| --- | --- |")
    lines.append(f"| **Deepgram transcript** | {row['deepgram_transcript']} |")
    lines.append(f"| **Genesys transcript** | {row['genesys_transcript']} |")
    lines.append(f"| **Text similarity** | {row['similarity']:.1%} |")
    lines.append(f"| **Similarity gap** | {row['similarity_gap']:.1%} |")
    lines.append(f"| **Deepgram confidence** | {row['deepgram_confidence']:.1%} |")
    lines.append(f"| **Genesys confidence** | {row['genesys_confidence']:.1%} |")
    lines.append(f"| **Channel** | {row['channel']} |")
    lines.append("")

OUTPUT_FILE.write_text("\n".join(lines))
print(f"Written {len(top_gaps)} examples to {OUTPUT_FILE}")

Written 25 examples to /Users/xnxn040/PycharmProjects/notifications-spike/analysis_results/text_similarity_gap_examples.md
